# 07: Hidden State構造の可視化 — PCA予備分析 + toorPIA

## 目的
v1データ（200サンプル × 3072次元）に対し、3エージェント×2ラウンドの構造が見えるかを確認する。

## 2部構成
- **Part 1 (Cell 1-5): PCA予備分析** — GPU不要・API不要・ローカル完結
- **Part 2 (Cell 6-): toorPIA可視化** — Part 1で構造が確認できた場合のみ実行

## 判断基準
| 問い | 指標 | 閾値 |
|------|------|------|
| Q1: エージェント分離 | シルエットスコア (agent_id) | > 0.1 で分離あり |
| Q2: ラウンド変化 | 移動距離の順列検定 | p < 0.05 で有意 |

## データ構造
- H: (200, 3072) = 100問 × 2ラウンド
- 3072 = Agent0[0:1024] ‖ Agent1[1024:2048] ‖ Agent2[2048:3072]
- metadata: example_idx, round, question, answer, responses

In [ ]:
# === Cell 1: セットアップ ===
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('thoughtcomm'):
        !git clone https://github.com/AUMEZAK/thoughtcomm.git
    %cd thoughtcomm
    from google.colab import drive
    drive.mount('/content/drive')

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

DRIVE_DIR = '/content/drive/MyDrive/thoughtcomm_checkpoints/'
print('Setup done.')

In [ ]:
# === Cell 2: データ読み込み + エージェント分割 ===
math_path = os.path.join(DRIVE_DIR, 'Qwen3-0.6B_math', 'hidden_states.pt')

ckpt = torch.load(math_path, map_location='cpu')
if 'H' in ckpt:
    H = ckpt['H'].float()
elif 'H_list' in ckpt:
    H = torch.stack(ckpt['H_list']).float()
meta = ckpt.get('metadata', [])

n_samples, n_h = H.shape
hidden_size = n_h // 3  # 1024
H_np = H.numpy()

# メタデータ展開
rounds = np.array([m.get('round', 0) for m in meta])
questions = np.array([m.get('example_idx', 0) for m in meta])

# エージェント分割: (200, 3072) → (600, 1024) + ラベル
H_agents = np.vstack([H_np[:, i*hidden_size:(i+1)*hidden_size] for i in range(3)])
agent_ids = np.repeat([0, 1, 2], n_samples)
round_ids = np.tile(rounds, 3)
question_ids = np.tile(questions, 3)

print(f'H shape: {H.shape} → エージェント分割後: {H_agents.shape}')
print(f'Rounds: {sorted(set(rounds))} | Questions: {len(set(questions))}問')
print(f'Round 0: {(rounds==0).sum()}サンプル, Round 1: {(rounds==1).sum()}サンプル')

In [ ]:
# === Cell 3: PCA散布図 — Q1(エージェント分離) + Q2(ラウンド変化) ===
# 共通PCA空間に600サンプルを射影して構造を確認

H_centered = H_agents - H_agents.mean(axis=0)
pca = PCA(n_components=3)
X_pca = pca.fit_transform(H_centered)
var_ratio = pca.explained_variance_ratio_

colors_agent = ['tab:blue', 'tab:orange', 'tab:green']
colors_round = ['tab:red', 'tab:purple']
markers_round = ['o', 's']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# (A) エージェント色分け → Q1
for a in range(3):
    mask = agent_ids == a
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_agent[a],
                    alpha=0.4, s=20, label=f'Agent {a}')
axes[0].set_title('(A) Agent color — Q1: 分離するか？')
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[0].legend()

# (B) ラウンド色分け → Q2
for r in [0, 1]:
    mask = round_ids == r
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_round[r],
                    alpha=0.4, s=20, label=f'Round {r}')
axes[1].set_title('(B) Round color — Q2: 移動するか？')
axes[1].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[1].legend()

# (C) Agent × Round（6グループ）
for a in range(3):
    for r in [0, 1]:
        mask = (agent_ids == a) & (round_ids == r)
        axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_agent[a],
                        marker=markers_round[r], alpha=0.4, s=25,
                        label=f'Agent {a} / Round {r}')
axes[2].set_title('(C) Agent × Round')
axes[2].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[2].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[2].legend(fontsize=8, ncol=2)

plt.tight_layout()
os.makedirs('results/toorpia', exist_ok=True)
plt.savefig('results/toorpia/pca_agent_round_scatter.png', dpi=150)
plt.show()

# --- PC1-PC3, PC2-PC3 も確認 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for a in range(3):
    mask = agent_ids == a
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 2], c=colors_agent[a],
                    alpha=0.4, s=20, label=f'Agent {a}')
    axes[1].scatter(X_pca[mask, 1], X_pca[mask, 2], c=colors_agent[a],
                    alpha=0.4, s=20, label=f'Agent {a}')
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[0].set_ylabel(f'PC3 ({var_ratio[2]:.1%})')
axes[0].set_title('Agent color: PC1 vs PC3')
axes[0].legend()
axes[1].set_xlabel(f'PC2 ({var_ratio[1]:.1%})')
axes[1].set_ylabel(f'PC3 ({var_ratio[2]:.1%})')
axes[1].set_title('Agent color: PC2 vs PC3')
axes[1].legend()
plt.tight_layout()
plt.savefig('results/toorpia/pca_agent_pc13_pc23.png', dpi=150)
plt.show()

print(f'PCA explained variance: PC1={var_ratio[0]:.1%}, PC2={var_ratio[1]:.1%}, PC3={var_ratio[2]:.1%}')

In [ ]:
# === Cell 4: Round 0→1 移動ベクトル分析 ===
# 同一問題・同一エージェントのペアを結んで移動を定量化

unique_questions = sorted(set(questions))
move_vectors = []  # PCA空間での移動ベクトル
move_dists = []    # PCA空間での移動距離
move_dists_orig = []  # 原空間での移動距離

for q in unique_questions:
    r0_idx = np.where((questions == q) & (rounds == 0))[0]
    r1_idx = np.where((questions == q) & (rounds == 1))[0]
    if len(r0_idx) != 1 or len(r1_idx) != 1:
        continue
    r0, r1 = r0_idx[0], r1_idx[0]
    for a in range(3):
        # PCA空間（エージェント分割後のインデックス）
        idx0 = a * n_samples + r0
        idx1 = a * n_samples + r1
        vec = X_pca[idx1] - X_pca[idx0]
        move_vectors.append(vec)
        move_dists.append(np.linalg.norm(vec))
        # 原空間
        h0 = H_agents[idx0]
        h1 = H_agents[idx1]
        move_dists_orig.append(np.linalg.norm(h1 - h0))

move_vectors = np.array(move_vectors)
move_dists = np.array(move_dists)
move_dists_orig = np.array(move_dists_orig)

print(f'=== Round 0→1 移動分析 ({len(move_dists)} ペア) ===')
print(f'PCA空間: mean={move_dists.mean():.3f}, std={move_dists.std():.3f}, max={move_dists.max():.3f}')
print(f'原空間:  mean={move_dists_orig.mean():.2f}, std={move_dists_orig.std():.2f}')

# 方向の一貫性: 移動ベクトル間のコサイン類似度
from itertools import combinations
cos_sims = []
for i, j in combinations(range(len(move_vectors)), 2):
    vi, vj = move_vectors[i], move_vectors[j]
    ni, nj = np.linalg.norm(vi), np.linalg.norm(vj)
    if ni > 1e-8 and nj > 1e-8:
        cos_sims.append(np.dot(vi, vj) / (ni * nj))
cos_sims = np.array(cos_sims)
print(f'方向一貫性 (cos類似度): mean={cos_sims.mean():.4f}, std={cos_sims.std():.4f}')
print(f'  ランダム方向ならmean≈0, 一貫した方向ならmean>0')

# 順列検定: ラウンドラベルをシャッフルして移動距離を比較
np.random.seed(42)
n_perm = 1000
perm_means = []
for _ in range(n_perm):
    # ラウンドラベルをシャッフル
    perm_rounds = rounds.copy()
    for q in unique_questions:
        q_mask = questions == q
        perm_rounds[q_mask] = np.random.permutation(perm_rounds[q_mask])
    # シャッフル後の移動距離を計算
    perm_dists = []
    for q in unique_questions:
        r0_idx = np.where((questions == q) & (perm_rounds == 0))[0]
        r1_idx = np.where((questions == q) & (perm_rounds == 1))[0]
        if len(r0_idx) != 1 or len(r1_idx) != 1:
            continue
        r0, r1 = r0_idx[0], r1_idx[0]
        for a in range(3):
            idx0 = a * n_samples + r0
            idx1 = a * n_samples + r1
            perm_dists.append(np.linalg.norm(X_pca[idx1] - X_pca[idx0]))
    perm_means.append(np.mean(perm_dists))

p_value = np.mean(np.array(perm_means) <= move_dists.mean())
print(f'\n順列検定 (n={n_perm}):')
print(f'  実際の平均移動距離:     {move_dists.mean():.4f}')
print(f'  シャッフル平均移動距離: {np.mean(perm_means):.4f} ± {np.std(perm_means):.4f}')
print(f'  p-value: {p_value:.4f} ({"有意" if p_value < 0.05 else "非有意"})')

# 可視化: PCA空間での矢印
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (A) 全ペアの矢印
for q in unique_questions:
    r0_idx = np.where((questions == q) & (rounds == 0))[0]
    r1_idx = np.where((questions == q) & (rounds == 1))[0]
    if len(r0_idx) != 1 or len(r1_idx) != 1:
        continue
    r0, r1 = r0_idx[0], r1_idx[0]
    for a in range(3):
        idx0 = a * n_samples + r0
        idx1 = a * n_samples + r1
        axes[0].annotate('', xy=X_pca[idx1, :2], xytext=X_pca[idx0, :2],
                         arrowprops=dict(arrowstyle='->', color=colors_agent[a], alpha=0.3, lw=0.8))
for a in range(3):
    axes[0].scatter([], [], c=colors_agent[a], label=f'Agent {a}')
axes[0].scatter([], [], c='gray', marker='>', label='Round 0→1')
axes[0].set_title('(A) Round 0→1 移動矢印 (PCA空間)')
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({var_ratio[1]:.1%})')
axes[0].legend()

# (B) 移動距離ヒストグラム vs シャッフル
axes[1].hist(perm_means, bins=30, alpha=0.6, label='Shuffled', color='gray')
axes[1].axvline(move_dists.mean(), color='red', linewidth=2, label=f'Actual (p={p_value:.3f})')
axes[1].set_title('(B) 順列検定: 移動距離')
axes[1].set_xlabel('Mean distance (PCA space)')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/toorpia/pca_round_movement.png', dpi=150)
plt.show()

In [ ]:
# === Cell 5: PCA予備分析サマリー + 判断 ===
import json

# Q1: エージェント分離（シルエットスコア）
sil_score = silhouette_score(X_pca, agent_ids)
print('=' * 60)
print('Part 1: PCA予備分析 結果サマリー')
print('=' * 60)

print(f'\n--- Q1: エージェント分離 ---')
print(f'シルエットスコア (agent_id): {sil_score:.4f}')
if sil_score > 0.1:
    q1_result = 'YES — 弱いが分離構造あり'
    print(f'  → {q1_result}')
    print(f'  PCA上位成分にエージェント固有の分散が含まれている')
else:
    q1_result = 'NO — PCA空間で分離なし'
    print(f'  → {q1_result}')
    print(f'  エージェント差異はPCA上位成分には現れない')
    print(f'  （cos=0.78の差異は低次成分に分散している可能性）')

print(f'\n--- Q2: ラウンド変化 ---')
print(f'順列検定 p-value: {p_value:.4f}')
print(f'方向一貫性 (cos): {cos_sims.mean():.4f}')
if p_value < 0.05:
    q2_result = 'YES — debate効果がhidden stateに反映'
    print(f'  → {q2_result}')
else:
    q2_result = 'NO — 統計的に有意な移動なし'
    print(f'  → {q2_result}')
    print(f'  debateの影響はhidden stateレベルで微弱')

print(f'\n--- 判断 ---')
if sil_score > 0.1 or p_value < 0.05:
    verdict = 'toorPIAで詳細分析に進む価値あり → Part 2を実行'
else:
    verdict = '同一モデル3コピーでは構造が見えない → 異なるモデルでの実験を優先'
print(f'  {verdict}')

# 結果保存
pca_summary = {
    'q1_silhouette': float(sil_score),
    'q1_result': q1_result,
    'q2_p_value': float(p_value),
    'q2_direction_cos_mean': float(cos_sims.mean()),
    'q2_move_dist_mean': float(move_dists.mean()),
    'q2_result': q2_result,
    'verdict': verdict,
    'pca_variance_ratio': [float(v) for v in var_ratio],
}
with open('results/toorpia/07_pca_summary.json', 'w') as f:
    json.dump(pca_summary, f, indent=2)
print(f'\nSaved: results/toorpia/07_pca_summary.json')

In [ ]:
# ============================================================
# Part 2: toorPIA可視化（Part 1で構造が確認できた場合のみ実行）
# ============================================================
# Part 1の結果を確認してから以下を実行してください。
# Q1 or Q2が YES の場合 → 実行する価値あり
# 両方 NO の場合 → 異なるモデルでの実験（v6 Step 1.3）を優先

!pip install -q toorpia
from toorpia import toorPIA
import pandas as pd

TOORPIA_API_KEY = 'toorpia_8acfd4e501d10123bd18bf5398051394c37cb08b0546b4a0'
client = toorPIA(api_key=TOORPIA_API_KEY)
print('toorPIA client ready.')

In [ ]:
# === Cell 7: Map A — エージェント別 (600 × 1024d) → toorPIA ===
# Cell 2で作成済みの H_agents, agent_ids, round_ids, question_ids を使用

df_agents = pd.DataFrame(H_agents, columns=[f'dim_{i}' for i in range(hidden_size)])
df_agents['agent_id'] = agent_ids
df_agents['round_id'] = round_ids
df_agents['question_id'] = question_ids

csv_path_a = 'results/toorpia/map_a_agents_1024d.csv'
df_agents.to_csv(csv_path_a, index=False)
print(f'CSV saved: {csv_path_a} ({df_agents.shape})')

import time
print('Creating Map A: 600 samples × 1024 dims (agent-split)')
result_a = client.basemap_csvform(
    csv_path_a,
    drop_columns=['agent_id', 'round_id', 'question_id'],
    label='ThoughtComm math: agent-split 1024d',
    tag='thoughtcomm-hidden-state',
)
print(f'Map A: mapNo={result_a["mapNo"]}, shareUrl={result_a["shareUrl"]}')
time.sleep(2)

# 可視化
xy = result_a['xyData']
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for a in range(3):
    mask = agent_ids == a
    axes[0].scatter(xy[mask, 0], xy[mask, 1], c=colors_agent[a],
                    alpha=0.5, s=20, label=f'Agent {a}')
axes[0].set_title('Map A (toorPIA): Agent color')
axes[0].legend()

for r in [0, 1]:
    mask = round_ids == r
    axes[1].scatter(xy[mask, 0], xy[mask, 1], c=colors_round[r],
                    alpha=0.5, s=20, label=f'Round {r}')
axes[1].set_title('Map A (toorPIA): Round color')
axes[1].legend()

for a in range(3):
    for r in [0, 1]:
        mask = (agent_ids == a) & (round_ids == r)
        axes[2].scatter(xy[mask, 0], xy[mask, 1], c=colors_agent[a],
                        marker=markers_round[r], alpha=0.5, s=25,
                        label=f'Agent {a} / Round {r}')
axes[2].set_title('Map A (toorPIA): Agent × Round')
axes[2].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('results/toorpia/map_a_scatter.png', dpi=150)
plt.show()

In [ ]:
# === Cell 8: Map B — 全体 (200 × 3072d) + Round移動矢印 ===
rows_b = []
for idx in range(n_samples):
    m = meta[idx]
    row = {f'dim_{i}': H_np[idx, i] for i in range(n_h)}
    row['round_id'] = m['round']
    row['question_id'] = m['example_idx']
    rows_b.append(row)

df_full = pd.DataFrame(rows_b)
csv_path_b = 'results/toorpia/map_b_full_3072d.csv'
df_full.to_csv(csv_path_b, index=False)

print('Creating Map B: 200 samples × 3072 dims (full concatenated)')
result_b = client.basemap_csvform(
    csv_path_b,
    drop_columns=['round_id', 'question_id'],
    label='ThoughtComm math: full 3072d',
    tag='thoughtcomm-hidden-state',
)
print(f'Map B: mapNo={result_b["mapNo"]}, shareUrl={result_b["shareUrl"]}')
time.sleep(2)

# 可視化
xy_b = result_b['xyData']
rounds_b = df_full['round_id'].values
questions_b = df_full['question_id'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for r in sorted(set(rounds_b)):
    mask = rounds_b == r
    axes[0].scatter(xy_b[mask, 0], xy_b[mask, 1], alpha=0.5, s=30, label=f'Round {int(r)}')
axes[0].set_title('Map B (toorPIA): Round color (200 × 3072d)')
axes[0].legend()

# 同一問題のRound 0→Round 1を矢印で結ぶ
for q in sorted(set(questions_b)):
    r0_mask = (questions_b == q) & (rounds_b == 0)
    r1_mask = (questions_b == q) & (rounds_b == 1)
    if r0_mask.sum() == 1 and r1_mask.sum() == 1:
        x0, y0 = xy_b[r0_mask][0]
        x1, y1 = xy_b[r1_mask][0]
        axes[1].annotate('', xy=(x1, y1), xytext=(x0, y0),
                         arrowprops=dict(arrowstyle='->', color='gray', alpha=0.3))
        axes[1].scatter(x0, y0, c='tab:red', s=15, alpha=0.6)
        axes[1].scatter(x1, y1, c='tab:purple', s=15, alpha=0.6)

axes[1].set_title('Map B (toorPIA): Round 0→1 movement per question')
axes[1].scatter([], [], c='tab:red', s=30, label='Round 0')
axes[1].scatter([], [], c='tab:purple', s=30, label='Round 1')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/toorpia/map_b_scatter.png', dpi=150)
plt.show()

# 移動距離統計
dists_b = []
for q in sorted(set(questions_b)):
    r0_mask = (questions_b == q) & (rounds_b == 0)
    r1_mask = (questions_b == q) & (rounds_b == 1)
    if r0_mask.sum() == 1 and r1_mask.sum() == 1:
        dists_b.append(np.linalg.norm(xy_b[r0_mask][0] - xy_b[r1_mask][0]))
dists_b = np.array(dists_b)
print(f'\ntoorPIA座標での移動距離: mean={dists_b.mean():.3f}, std={dists_b.std():.3f}, max={dists_b.max():.3f}')

In [ ]:
# === Cell 9: Map C — PCA64次元圧縮版 (600 × 64d) ===
dim_cols = [f'dim_{i}' for i in range(hidden_size)]
X = df_agents[dim_cols].values
X_centered = X - X.mean(axis=0)

pca64 = PCA(n_components=64)
X_pca64 = pca64.fit_transform(X_centered)
print(f'PCA: {hidden_size}d → 64d, explained variance = {pca64.explained_variance_ratio_.sum():.1%}')

df_pca = pd.DataFrame(X_pca64, columns=[f'pc_{i}' for i in range(64)])
df_pca['agent_id'] = agent_ids
df_pca['round_id'] = round_ids
df_pca['question_id'] = question_ids

csv_path_c = 'results/toorpia/map_c_pca64d.csv'
df_pca.to_csv(csv_path_c, index=False)

print('Creating Map C: 600 samples × 64 dims (PCA-reduced agent-split)')
result_c = client.basemap_csvform(
    csv_path_c,
    drop_columns=['agent_id', 'round_id', 'question_id'],
    label='ThoughtComm math: PCA64d agent-split',
    tag='thoughtcomm-hidden-state',
)
print(f'Map C: mapNo={result_c["mapNo"]}, shareUrl={result_c["shareUrl"]}')
time.sleep(2)

# 可視化
xy_c = result_c['xyData']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for a in range(3):
    mask = agent_ids == a
    axes[0].scatter(xy_c[mask, 0], xy_c[mask, 1], c=colors_agent[a],
                    alpha=0.5, s=20, label=f'Agent {a}')
axes[0].set_title('Map C (PCA64d → toorPIA): Agent color')
axes[0].legend()

for a in range(3):
    for r in [0, 1]:
        mask = (agent_ids == a) & (round_ids == r)
        axes[1].scatter(xy_c[mask, 0], xy_c[mask, 1], c=colors_agent[a],
                        marker=markers_round[r], alpha=0.5, s=25,
                        label=f'Agent {a} / Round {r}')
axes[1].set_title('Map C (PCA64d → toorPIA): Agent × Round')
axes[1].legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('results/toorpia/map_c_scatter.png', dpi=150)
plt.show()

In [ ]:
# === Cell 10: 全結果サマリー ===

print('=' * 70)
print('Notebook 07: Hidden State構造の可視化 — 全結果')
print('=' * 70)

print(f'\n=== Part 1: PCA予備分析 ===')
print(f'Q1 エージェント分離 (silhouette): {sil_score:.4f} → {q1_result}')
print(f'Q2 ラウンド変化 (p-value):        {p_value:.4f} → {q2_result}')
print(f'   方向一貫性 (cos):              {cos_sims.mean():.4f}')
print(f'判断: {verdict}')

print(f'\n=== Part 2: toorPIA可視化 ===')
print(f'Map A (agent-split 600×1024d): mapNo={result_a["mapNo"]}')
print(f'  {result_a["shareUrl"]}')
print(f'Map B (full 200×3072d):        mapNo={result_b["mapNo"]}')
print(f'  {result_b["shareUrl"]}')
print(f'  Round移動距離: mean={dists_b.mean():.3f}')
print(f'Map C (PCA64d 600×64d):        mapNo={result_c["mapNo"]}')
print(f'  {result_c["shareUrl"]}')

# 全結果保存
summary = {
    'part1_pca': pca_summary,
    'part2_toorpia': {
        'map_a': {'mapNo': result_a['mapNo'], 'shareUrl': result_a['shareUrl'],
                  'desc': 'agent-split 600x1024d'},
        'map_b': {'mapNo': result_b['mapNo'], 'shareUrl': result_b['shareUrl'],
                  'desc': 'full 200x3072d', 'round_move_dist_mean': float(dists_b.mean())},
        'map_c': {'mapNo': result_c['mapNo'], 'shareUrl': result_c['shareUrl'],
                  'desc': 'PCA64d agent-split 600x64d'},
    }
}
with open('results/toorpia/07_full_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: results/toorpia/07_full_summary.json')